In [1]:
import pandas as pd
import numpy as np
import folium
import os

In [2]:
gps = pd.read_csv(
    "../data/processed/gps_anomalies.csv"
)

priority = pd.read_csv(
    "../data/processed/search_priority_scores.csv"
)

route = pd.read_csv(
    "../data/processed/predicted_route.csv"
)

cases = pd.read_csv(
    "../data/synthetic/missing_person_cases.csv"
)

print("GPS:", gps.shape)
print("Priority:", priority.shape)
print("Route:", route.shape)
print("Cases:", cases.shape)

GPS: (24730077, 23)
Priority: (7, 10)
Route: (5, 2)
Cases: (100, 15)


In [3]:
case = cases.iloc[0]

print("Case ID:", case["Case_ID"])
print(
    "Last Latitude:",
    case["Last_Latitude"]
)
print(
    "Last Longitude:",
    case["Last_Longitude"]
)

Case ID: MP-2026-001
Last Latitude: 39.980525
Last Longitude: 116.44944


In [4]:
last_lat = float(
    case["Last_Latitude"]
)

last_lon = float(
    case["Last_Longitude"]
)

print(
    "Last known location:",
    last_lat,
    last_lon
)

Last known location: 39.980525 116.44944


In [5]:
m = folium.Map(
    location=[
        last_lat,
        last_lon
    ],
    zoom_start=12
)

print("Base map created!")

Base map created!


In [6]:
folium.Marker(
    location=[
        last_lat,
        last_lon
    ],
    popup=(
        f"Case: {case['Case_ID']}<br>"
        "Last Known Location"
    ),
    tooltip="Last Known Location"
).add_to(m)

In [7]:
valid_gps = gps[
    gps["cluster"] != -1
].copy()

area_centers = (
    valid_gps
    .groupby("cluster")
    .agg(
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean"),
        visits=("cluster", "size")
    )
    .reset_index()
)

area_centers.head()

,cluster,latitude,longitude,visits
0,0,42.367534,117.556758,785358
1,1,39.885085,116.514713,5487320
2,2,30.193778,114.729449,1160492
3,3,44.775164,6.829885,170423
4,4,20.073562,111.095783,591782


In [8]:
for _, row in area_centers.iterrows():

    folium.CircleMarker(
        location=[
            row["latitude"],
            row["longitude"]
        ],
        radius=5,
        popup=(
            f"Area: {int(row['cluster'])}<br>"
            f"Visits: {int(row['visits'])}"
        ),
        tooltip=(
            f"Frequently Visited Area "
            f"{int(row['cluster'])}"
        ),
        fill=True
    ).add_to(m)

In [9]:
top5 = priority.head(5).copy()

top5

,Rank,Area,ML_Score,Historical_Score,Route_Score,Distance_Score,Time_Score,Anomaly_Score,Priority_Score,Priority
0,1,9,100.0,100.00,0.0,99.90,100.00,0.14,75.00,High
1,2,1,0.0,37.47,0.0,99.87,48.16,0.42,27.33,Low
2,3,0,0.0,5.36,0.0,96.99,7.01,5.77,16.90,Low
3,4,6,0.0,4.67,0.0,98.30,4.49,4.07,16.54,Low
4,5,2,0.0,7.92,0.0,88.23,8.01,6.52,16.27,Low


In [10]:
top5 = top5.merge(
    area_centers,
    left_on="Area",
    right_on="cluster",
    how="left"
)

top5

,Rank,Area,ML_Score,Historical_Score,Route_Score,Distance_Score,Time_Score,Anomaly_Score,Priority_Score,Priority,cluster,latitude,longitude,visits
0,1,9,100.0,100.00,0.0,99.90,100.00,0.14,75.00,High,9,40.009068,116.343671,14645638
1,2,1,0.0,37.47,0.0,99.87,48.16,0.42,27.33,Low,1,39.885085,116.514713,5487320
2,3,0,0.0,5.36,0.0,96.99,7.01,5.77,16.90,Low,0,42.367534,117.556758,785358
3,4,6,0.0,4.67,0.0,98.30,4.49,4.07,16.54,Low,6,38.561120,116.251663,683587
4,5,2,0.0,7.92,0.0,88.23,8.01,6.52,16.27,Low,2,30.193778,114.729449,1160492


In [11]:
for _, row in top5.iterrows():

    if pd.isna(row["latitude"]):
        continue

    folium.Marker(
        location=[
            row["latitude"],
            row["longitude"]
        ],
        popup=(
            f"Predicted Area: "
            f"{int(row['Area'])}<br>"
            f"Probability Score: "
            f"{row['ML_Score']:.2f}<br>"
            f"Priority Score: "
            f"{row['Priority_Score']:.2f}<br>"
            f"Priority: "
            f"{row['Priority']}"
        ),
        tooltip=(
            f"Predicted Area "
            f"{int(row['Area'])}"
        )
    ).add_to(m)

In [12]:
for _, row in top5.iterrows():

    if pd.isna(row["latitude"]):
        continue

    folium.Circle(
        location=[
            row["latitude"],
            row["longitude"]
        ],
        radius=300,
        popup=(
            f"Search Priority Area "
            f"{int(row['Area'])}<br>"
            f"Score: "
            f"{row['Priority_Score']:.2f}<br>"
            f"Priority: "
            f"{row['Priority']}"
        ),
        tooltip="Search Priority Area",
        fill=True
    ).add_to(m)

In [13]:
anomalies = gps[
    gps["anomaly"] == -1
].copy()

anomalies = anomalies.head(200)

print(
    "Anomalies plotted:",
    len(anomalies)
)

Anomalies plotted: 200


In [14]:
for _, row in anomalies.iterrows():

    folium.CircleMarker(
        location=[
            row["latitude"],
            row["longitude"]
        ],
        radius=3,
        popup=(
            "Statistical Anomaly<br>"
            f"Score: "
            f"{row['anomaly_score']:.4f}"
        ),
        tooltip="Anomalous Movement",
        fill=True
    ).add_to(m)

In [15]:
route

,Step,Area
0,0,0.0
1,1,9.0
2,2,1.0
3,3,9.0
4,4,1.0


In [16]:
route_map = route.merge(
    area_centers,
    left_on="Area",
    right_on="cluster",
    how="left"
)

route_map

,Step,Area,cluster,latitude,longitude,visits
0,0,0.0,0,42.367534,117.556758,785358
1,1,9.0,9,40.009068,116.343671,14645638
2,2,1.0,1,39.885085,116.514713,5487320
3,3,9.0,9,40.009068,116.343671,14645638
4,4,1.0,1,39.885085,116.514713,5487320


In [17]:
route_coordinates = []

for _, row in route_map.iterrows():

    if pd.isna(row["latitude"]):
        continue

    route_coordinates.append([
        row["latitude"],
        row["longitude"]
    ])

print(
    "Route points:",
    len(route_coordinates)
)

Route points: 5


In [18]:
if len(route_coordinates) >= 2:

    folium.PolyLine(
        route_coordinates,
        weight=5,
        popup="Probable Movement Route",
        tooltip="Probable Route"
    ).add_to(m)

In [19]:
for i, (_, row) in enumerate(
    route_map.iterrows(),
    start=1
):

    if pd.isna(row["latitude"]):
        continue

    folium.Marker(
        location=[
            row["latitude"],
            row["longitude"]
        ],
        popup=(
            f"Route Step: {i}<br>"
            f"Area: {int(row['Area'])}"
        ),
        tooltip=f"Route Step {i}"
    ).add_to(m)

In [20]:
folium.LayerControl().add_to(m)

In [21]:
m

In [22]:
os.makedirs(
    "../data/processed/maps",
    exist_ok=True
)

In [23]:
map_path = (
    "../data/processed/maps/"
    "case_investigation_map.html"
)

m.save(map_path)

print(
    "Interactive map saved:"
)

print(map_path)

Interactive map saved:
../data/processed/maps/case_investigation_map.html


In [24]:
print(
    os.path.exists(map_path)
)

True


In [25]:
print(
    "========== INTERACTIVE MAP =========="
)

print(
    "Case ID:",
    case["Case_ID"]
)

print(
    "Last Known Location:",
    last_lat,
    last_lon
)

print(
    "Predicted Areas:",
    len(top5)
)

print(
    "Anomalies Displayed:",
    len(anomalies)
)

print(
    "Route Points:",
    len(route_coordinates)
)

print(
    "Map saved successfully!"
)

========== INTERACTIVE MAP ==========
Case ID: MP-2026-001
Last Known Location: 39.980525 116.44944
Predicted Areas: 5
Anomalies Displayed: 200
Route Points: 5
Map saved successfully!


In [26]:
import sys
!{sys.executable} -m pip install streamlit streamlit-folium folium